### 실습 환경 준비

In [1]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings as RagasHuggingFaceEmbeddings
from dotenv import load_dotenv

load_dotenv("../.env")
openai_client = AsyncOpenAI()

generator_llm = llm_factory(
    model="gpt-6-luna",
    client=openai_client,
    reasoning_effort="none",
    max_tokens=4096,
)

generator_embeddings = RagasHuggingFaceEmbeddings(
    model="intfloat/multilingual-e5-base",
    device="cuda",
    normalize_embeddings=True,
    prompts={"passage": "passage: "},
    default_prompt_name="passage"
)

c:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1530.28it/s]
Default prompt name is set to 'passage'. This prompt applies unless overridden by `prompt`, `prompt_name`, or a method-specific prompt, including an empty prompt.


### 7.5 LangSmith와 Ragas를 활용한 오프라인 평가 구현

In [2]:
import nest_asyncio

from typing import Any
from langchain_core.documents import Document
from langsmith import Client
from langsmith.schemas import Example, Run

nest_asyncio.apply()

class RagasMetricEvaluator:
    def __init__(self, metric, input_fields: list[str]):
        self.metric = metric
        self.input_fields = input_fields

    async def evaluate(self, run: Run, example: Example) -> dict[str, Any]:
        # 실제 검색 결과에서 본문 추출
        context_strs = [
            doc.page_content if isinstance(doc, Document) else doc
            for doc in run.outputs.get("contexts", [])
        ]

        # 평가에 사용할 데이터 구성
        data = {
            "user_input": example.inputs["question"],
            "response": run.outputs["answer"],
            "retrieved_contexts": context_strs,
            "reference": (example.outputs or {}).get("answer"),
        }

        # 매트릭에 필요한 입력 선택
        metric_inputs = {field: data[field] for field in self.input_fields}

        # 비동기식 점수 계산
        result = await self.metric.ascore(**metric_inputs)

        return {"key": self.metric.name, "score": float(result.value)}

In [3]:
# 위의 RagasMetricEvaluator를 사용해 Ragas의 평가 메트릭을 준비
from ragas.metrics.collections import ContextPrecision, AnswerRelevancy

# 기존 GPT 모델을 연결한 Context precision 메트릭
context_precision_metric = ContextPrecision(llm=generator_llm)
# 기존 GPT 모델과 E5를 연결한 Answer relevancy 메트릭
answer_relevancy_metric = AnswerRelevancy(llm=generator_llm, embeddings=generator_embeddings)

# LangSmith에 전달할 평가 함수 목록
evaluators = [
    RagasMetricEvaluator(metric=context_precision_metric, input_fields=["user_input", "reference", "retrieved_contexts"]).evaluate,
    RagasMetricEvaluator(metric=answer_relevancy_metric, input_fields=["user_input", "response"]).evaluate
]


### 추론 함수 구현

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"prompt": "passage: ", "normalize_embeddings": True},
    query_encode_kwargs={"prompt": "query: ", "normalize_embeddings": True}
)

db = Chroma(
    persist_directory="../ch06_advanced_rag/chroma_db_e5",
    collection_name="langchain-docs",
    embedding_function=embeddings,
    create_collection_if_not_exists=False
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1076.29it/s]


In [5]:
# 다음으로 RAG의 간단한 Chain을 구현
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template('''
다음 문맥만을 고려해 질문에 답하세요.
문맥: """{context}"""
질문: {question}
''')

model = ChatOpenAI(model="gpt-6-luna")
retriever = db.as_retriever()

chain = RunnableParallel(
    {
        "question": RunnablePassthrough(),
        "context": retriever,
    }
).assign(answer=prompt | model | StrOutputParser())

In [6]:
# LangSmith에서 평가에 사용할 추론 함수는 데이터셋에 저장한 형식의 dict를 받아 실제 실행 결과(run)를 dict로 반환하는 함수로 구현
# 데이터셋의 입력에서 question을 추출해 추론(RAG 등의 처리)을 수행하고, 실제 검색 결과나 답변을 반환하는 함수는 다음과 같다.
async def predict(inputs: dict[str, Any]) -> dict[str, Any]:
    question = inputs["question"]
    output = await chain.ainvoke(question)
    
    return {
        "contexts": output["context"],
        "answer": output["answer"]
    }

### 오프라인 평가 구현: 실행

In [7]:
from langsmith import Client

langsmith_client = Client()

results = await langsmith_client.aevaluate(
    predict,
    data="agent-book",
    evaluators=evaluators,
)

View the evaluation results for experiment: 'excellent-paper-61' at:
https://aws.smith.langchain.com/o/0c052df7-f25f-4689-a76f-88a0b8e6cae2/datasets/b8e09122-7060-40f5-8fcc-16b93c7a675b/compare?selectedSessions=085eb3fa-7a8b-47a5-9525-f523df1e01a1




6it [02:00, 20.16s/it]
